# 04 · Gemini reply prediction
Goal: final prompt for `api/gemini.py`, plus **top-6 hit rate** and **latency** (must fit the 3.5 s budget).
Run top to bottom. Output: `ml/results/gemini_eval.json`.

In [1]:
import os, time, json, re, statistics
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel
from google import genai
from google.genai import types

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"],
                      http_options=types.HttpOptions(timeout=15000))   # generous here; gemini.py uses 3.5 s
OUT = ROOT / "ml" / "results"; OUT.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT, "| model in .env:", os.environ.get("GEMINI_MODEL"))

repo root: /Users/yashvarpe/Desktop/UBHack/blink-talk | model in .env: gemini-3.5-flash


## Profile + system instruction

In [2]:
PROFILE = {
    "name": "Maya",
    "caregivers": ["Alex (husband)", "Priya (nurse)"],
    "needs": ["water often", "help changing position", "suction sometimes"],
    "preferences": ["likes apple juice", "loves the Beatles", "daughter Sara calls every evening", "likes the blinds open"],
    "commonPhrases": ["Thank you", "I love you", "Yes", "No", "Not now", "Please wait"],
}

SYSTEM = """You suggest spoken replies for a person who cannot speak and picks a reply with their eyes.
Write as that person, in the first person. 

Rules:
- Return exactly 6 replies, each at most 6 words, plain words, no emojis, no quotes.
- Replies 1-2 must directly answer what the caregiver just said.
  For a yes/no question: reply 1 starts with "Yes", reply 2 starts with "No".
  For a choice ("A or B?"): replies 1-2 pick each option (the profile's preferred option first).
  For an open question ("anything?", "how are you?"): lead with the profile's needs.
- Replies 3-5 stay on the caregiver's topic first; use profile needs, people and preferences only when they fit.
  Do not mention the same preference in every answer.
- Reply 6 is always a short way to defer, like "Not now, thanks" or "Later, please".
- Never give medical advice, diagnoses or medication instructions.
- If the caregiver said nothing, suggest useful openers for the time of day.
- Only mention people who are in the profile."""
class Suggestions(BaseModel):
    phrases: list[str]

## Prompt builder, padding, call (this becomes `api/gemini.py`)

In [3]:
DEFAULTS = {
    "morning":   ["Good morning", "I slept well", "I didn't sleep well", "I'm hungry", "Open the blinds, please", "I need water"],
    "afternoon": ["I need water", "I'm tired", "Can you move me?", "I'm hungry", "Turn on the TV", "I'm comfortable"],
    "evening":   ["I'm tired", "Is Sara calling tonight?", "I need water", "Can you move me?", "Turn the light off", "Good night"],
}
def time_defaults(local_time):
    h = int(local_time.split(":")[0])
    return DEFAULTS["morning"] if h < 12 else DEFAULTS["afternoon"] if h < 17 else DEFAULTS["evening"]

def finalize(phrases, profile, local_time):
    """dedupe, keep <= 8 words, pad to exactly 6"""
    out, seen = [], set()
    for p in list(phrases) + profile.get("commonPhrases", []) + time_defaults(local_time):
        p = str(p).strip().strip('"').strip()
        key = re.sub(r"\W+", " ", p.lower()).strip()
        if p and key and key not in seen and len(p.split()) <= 8:
            seen.add(key); out.append(p)
        if len(out) == 6:
            break
    return out

def build_prompt(transcript, history, profile, local_time):
    lines = [f"Local time: {local_time}",
             f"User: {profile['name']}",
             f"Caregivers: {', '.join(profile['caregivers'])}",
             f"Needs: {', '.join(profile['needs'])}",
             f"Preferences: {', '.join(profile['preferences'])}"]
    if history:
        lines.append("Recent conversation (oldest first):")
        lines += [f"- {m['role']}: {m['text']}" for m in history[-8:]]
    lines.append(f'Caregiver just said: "{transcript}"' if transcript
                 else "The caregiver has not said anything. Suggest useful openers.")
    return "\n".join(lines)

THINK = {}   # model -> working thinking config
def thinking_candidates(model):
    if model.startswith("gemini-3"):
        return [types.ThinkingConfig(thinking_level="minimal"), types.ThinkingConfig(thinking_level="low")]
    return [types.ThinkingConfig(thinking_budget=0)]

def call_gemini(model, transcript, history=(), profile=PROFILE, local_time="18:30"):
    prompt = build_prompt(transcript, list(history), profile, local_time)
    cands = [THINK[model]] if model in THINK else thinking_candidates(model)
    last = None
    for tc in cands:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, response_mime_type="application/json",
                                          response_schema=Suggestions, temperature=0.4,
                                          max_output_tokens=300, thinking_config=tc)
        try:
            t = time.time()
            r = client.models.generate_content(model=model, contents=prompt, config=cfg)
            dt = time.time() - t
            THINK[model] = tc
            raw = r.parsed.phrases if r.parsed else json.loads(r.text)["phrases"]
            return finalize(raw, profile, local_time), dt
        except Exception as e:
            last = e
    raise last

phrases, dt = call_gemini("gemini-3.5-flash", "Do you want water or juice?")
print(f"{dt:.2f}s", phrases)

1.04s ['Juice, please.', 'Water would be great.', 'I prefer apple juice today.', 'Is it time for Sara?', 'Keep the blinds open please.', 'Neither right now, thanks.']


## Test set: 20 caregiver questions
`accept` = lowercase pieces of text; a test is a **hit** if any of the 6 phrases contains any of them (the intended reply is on screen).

In [4]:
TESTS = [
    ("Do you want water or juice?",                    ["juice"]),
    ("Are you in any pain right now?",                 ["yes"]),
    ("Are you cold?",                                  ["no"]),
    ("Do you want me to turn on the TV?",              ["yes"]),
    ("Do you want to sit up a bit?",                   ["yes", "sit up"]),
    ("How are you feeling today?",                     ["tired"]),
    ("Do you need the bathroom?",                      ["yes", "bathroom"]),
    ("Do you want me to read to you?",                 ["not now", "later"]),
    ("Is the music too loud?",                         ["too loud", "yes", "turn it down", "lower"]),
    ("Did you sleep well last night?",                 ["no", "not well", "didn't sleep"]),
    ("Who should I call?",                             ["sara"]),
    ("Do you want the blinds open or closed?",         ["open"]),
    ("Are you hungry?",                                ["yes", "hungry"]),
    ("Do you want your blanket?",                      ["yes", "blanket"]),
    ("Can I get you anything?",                        ["water"]),
    ("Is your position comfortable?",                  ["no", "move", "uncomfortable", "adjust"]),
    ("Want to listen to some music?",                  ["beatles"]),
    ("Should I turn off the light?",                   ["yes"]),
    ("Priya is here, do you want to see her?",         ["yes"]),
    ("Do you need suction?",                           ["yes", "suction"]),
]
def hit(phrases, accept):
    low = [p.lower() for p in phrases]
    return any(a in p for p in low for a in accept)
len(TESTS)

20

## Evaluate models (hit rate + latency)

In [5]:
MODELS = ["gemini-3.5-flash-lite"]
MODELS = list(dict.fromkeys(MODELS))   # dedupe, keep order
results = {}
for m in MODELS:
    try: call_gemini(m, "Hello")         # warm-up, not timed
    except Exception as e: print(m, "warm-up failed:", str(e)[:100]); continue
    rows = []
    for q, accept in TESTS:
        try:
            phrases, dt = call_gemini(m, q); ok = hit(phrases, accept)
        except Exception as e:
            phrases, dt, ok = [f"ERROR {str(e)[:80]}"], None, False
        rows.append({"q": q, "accept": accept, "phrases": phrases, "latency": dt, "hit": ok})
    lat = sorted(r["latency"] for r in rows if r["latency"] is not None)
    results[m] = {
        "hit_rate": sum(r["hit"] for r in rows) / len(rows),
        "p50_s": statistics.median(lat) if lat else None,
        "p95_s": lat[int(0.95 * (len(lat) - 1))] if lat else None,
        "over_3_5s": sum(l > 3.5 for l in lat),
        "thinking": str(THINK.get(m)),
        "rows": rows,
    }
    r = results[m]
    print(f"{m:24s} hit {r['hit_rate']:.0%}  p50 {r['p50_s']:.2f}s  p95 {r['p95_s']:.2f}s  >3.5s: {r['over_3_5s']}/{len(lat)}")

gemini-3.5-flash-lite    hit 95%  p50 0.82s  p95 0.93s  >3.5s: 0/20


## Misses (fix the prompt here, then re-run the cell above)

In [6]:
best = max(results, key=lambda m: (results[m]["hit_rate"], -(results[m]["p95_s"] or 99)))
print("best:", best)
for r in results[best]["rows"]:
    mark = "✓" if r["hit"] else "✗"
    print(f"{mark} {r['q']}\n   {r['phrases']}")

best: gemini-3.5-flash-lite
✓ Do you want water or juice?
   ['Juice please', 'Water please', 'Apple juice sounds good', 'My throat is dry', 'I am thirsty', 'Not now, thanks']
✓ Are you in any pain right now?
   ['Yes, I hurt a bit', 'No, I feel fine', 'Please help me move', 'I need some water', 'Can Sara call soon', 'Not now, thanks']
✓ Are you cold?
   ['Yes, I am cold', 'No, I am fine', 'Please close the window', 'Can I have water', 'Help me change position', 'Not now, thanks']
✓ Do you want me to turn on the TV?
   ['Yes, please', 'No, thank you', 'Listen to the Beatles', 'Open the blinds', 'Call my daughter Sara', 'Not now, thanks']
✓ Do you want to sit up a bit?
   ['Yes please help me sit', 'No I am fine lying down', 'I need help changing position', 'Let us leave the blinds open', 'Can Sara call soon please', 'Not now thanks']
✗ How are you feeling today?
   ['I need some water', 'I want help changing position', 'My throat needs suction', 'I am doing alright today', 'Let us list

## Save

In [8]:
summary = {m: {k: v for k, v in r.items() if k != "rows"} for m, r in results.items()}
json.dump({"best": best, "n_tests": len(TESTS), "summary": summary, "results": results,
           "system": SYSTEM}, open(OUT / "gemini_eval.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

{
  "gemini-3.5-flash-lite": {
    "hit_rate": 0.95,
    "p50_s": 0.8178631067276001,
    "p95_s": 0.9270951747894287,
    "over_3_5s": 0,
    "thinking": "include_thoughts=None thinking_budget=None thinking_level=<ThinkingLevel.MINIMAL: 'MINIMAL'>"
  }
}
